# Microsam Finetune Evaluation

Evaluate the finetuned microsam model on unseen data. Evaluate the result visually.

In [ ]:
%load_ext autoreload
%autoreload 2

## Models under evaluation

The base micro-sam LM model (before finetuning) and the Candida-finetuned model (after finetuning). `run_both_models` runs the AIS pipeline on the same images with each model, so every dataset below is evaluated on both without editing model paths by hand. Each model writes its own embedding cache and output masks, keyed by a model tag, so the two runs never overwrite each other.

In [ ]:
# A deepcopy of the per-dataset config is used per model so its settings (DIC shift,
# file patterns) stay intact and the two models never leak into each other.
from pathlib import Path
from copy import deepcopy
from image_processing_tools.image_class.micro_sam_pipeline import MicroSAMPipeline

DATA = Path("~/Projects/C_Albicans Thesis Project/7. Data").expanduser()
MODELS = {
    "before_finetune": dict(
        checkpoint_path=DATA / "model_checkpoints/micro_sam/models/vit_b_lm",
        decoder_checkpoint_path=DATA / "model_checkpoints/micro_sam/models/vit_b_lm_decoder",
    ),
    "after_finetune": dict(
        checkpoint_path=DATA / "microsam_finetune/final_model/vit_b_candida",
        decoder_checkpoint_path=DATA / "microsam_finetune/final_model/vit_b_candida_decoder",
    ),
}


def run_both_models(image_containers, base_config):
    """Run the AIS pipeline on `image_containers` once per model in MODELS.

    Returns {model_name: MicroSAMPipeline}.
    """
    pipelines = {}
    for model_name, ckpts in MODELS.items():
        cfg = deepcopy(base_config)
        cfg.update(ckpts)
        print(f"\n===== {model_name} =====")
        pipe = MicroSAMPipeline(image_containers, config=cfg)
        pipe.run()
        pipe.save_results()
        pipelines[model_name] = pipe
    return pipelines


## Model configurations

In [ ]:
from image_processing_tools.image_class.micro_sam_pipeline import load_config
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer

config_path = Path("./micro_sam_config.json")

config = load_config(config_path)
config['model_type'] = 'vit_b_lm'
config['segmentation_mode'] = 'automatic'

config['preprocessing']['resize_image'] = False
config['preprocessing']['max_dim'] = 1024
config['preprocessing']['quantization'] = "8bit"
config['preprocessing']['correct_DIC_shift'] = [0,0]
config['ndim'] = 2
config['tiling']['tile_shape'] = (512,512)
config['tiling']['halo'] = (64,64)

## Monoculture hyphae data from Eva

In [ ]:
from pathlib import Path
import matplotlib
# matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

search_path = (
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET161_ASH1Q670_02_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET155_ASH1Q670_04_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET147_ECE1Q670_06_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET146_ASH1Q670_07_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET146_ASH1Q670_02_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET145_ASH1Q670_02_CY5, DAPI",
    )

file_pattern = ("CROP_*.tif",
                "MAX_*",
                "SHARPEST*C1*.tif",
                "*DIC.tif",
                "MAX_CET145*")

# Find the files. The 'files' variable will hold the list of Path objects.
dic_files = []
for p in search_path:
    dic_files.extend(find_files_by_pattern(p, file_pattern[3], verbose=True))

dic_imgs = [ImageContainer(img,config) for img in dic_files]

In [ ]:
dic_imgs[0].config

In [ ]:
# Eva's monoculture hyphae data.
eva_mono = run_both_models(dic_imgs, config) 

## Load Wang's hyphae data

In [ ]:
search_path = (
    "~/A8/Data_Wang/260213_HWPA_DAPI/",
    )

file_pattern = ("CROP_*.tif",
                "MAX_*",
                "SHARPEST*C1*.tif",
                "*DIC.tif",
                "MAX_CET145*")

dic_files_wang = []
for p in search_path:
    dic_files_wang.extend(find_files_by_pattern(p, file_pattern[3], verbose=True))

dic_imgs_wang = [ImageContainer(img,config) for img in dic_files_wang]

In [ ]:
dic_imgs_wang

In [ ]:
# Wang's hyphae data.
wang_hyphae = run_both_models(dic_imgs_wang, config)

## Load Eva's coculture data

In [ ]:
search_path = (
    "~/A8/Data_Eva/Training data Chuyao/260415_Ca_Co_She3Mutants_Ca922_6h_F12_ECE1_cFOS/CET146_ECE1Q670_cFOSQ570_06_CY5, CFP, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/260415_Ca_Co_She3Mutants_Ca922_6h_F12_ECE1_cFOS/CET147_ECE1Q670_cFOSQ570_05_CY5, CFP, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/260415_Ca_Co_She3Mutants_Ca922_6h_F12_ECE1_cFOS/CET146_ECE1Q670_cFOSQ570_07_CY5, CFP, DAPI",
    )

file_pattern = ("MAX_C3_*.tif",
                "*DIC.tif",
                "MAX_C2_*.tif")

config['preprocessing']['correct_DIC_shift'] = [5,22]

dic_files_2 = []
fluo_files_2 = []
dapi_files_2 = []
for p in search_path:
    dic_files_2.extend(find_files_by_pattern(p, file_pattern[1], verbose=True))
    fluo_files_2.extend(find_files_by_pattern(p, file_pattern[2], verbose=True))
    dapi_files_2.extend(find_files_by_pattern(p, file_pattern[0], verbose=True))

In [ ]:
merged_imgs = [ImageContainer([di,f,da],config) for di,f,da in zip(dic_files_2,fluo_files_2,dapi_files_2)]
merged_imgs

In [ ]:
merged_imgs[0].display()

In [ ]:
# Eva's coculture data.
eva_cocu = run_both_models(merged_imgs, config)

## Roan's coculture data

In [ ]:
search_path = (
    "~/A8/Data_Roan/260612_Cocu_Mono_DyesTest/Cocu_CET155_24_Phal_Cell_Calc_06_CY5, FITC, DAPI",
    )

file_pattern = ("SHARPEST*_C3_*.tif",
                "*DIC.tif",
                "SHARPEST*_C1_*.tif")

config['preprocessing']['correct_DIC_shift'] = [5,22]

dic_files = []
phall_files = []
calco_files = []
for p in search_path:
    dic_files.extend(find_files_by_pattern(p, file_pattern[1], verbose=True))
    phall_files.extend(find_files_by_pattern(p, file_pattern[2], verbose=True))
    calco_files.extend(find_files_by_pattern(p, file_pattern[0], verbose=True))

merged_imgs_1 = [ImageContainer([d,p,c],config) for d,p,c in zip (dic_files, phall_files, calco_files)]

In [ ]:
merged_imgs_1[0].display()

In [ ]:
# Roan's coculture data.
roan_cocu = run_both_models(merged_imgs_1, config)

## Roan's another co-culture set, DIC only

In [ ]:
search_path = (
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/03/",
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/04/",
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/05/",
    )

file_pattern = ("CROP_*.tif",
                "MAX_*",
                "SHARPEST*C1*.tif",
                "*DIC.tif",
                "MAX_CET145*")

dic_files_roan = []
for p in search_path:
    dic_files_roan.extend(find_files_by_pattern(p, file_pattern[3], verbose=True))

dic_imgs_roan = [ImageContainer(img,config) for img in dic_files_roan]

In [ ]:
# Roan's DIC-only coculture data.
roan_dic = run_both_models(dic_imgs_roan, config)

## Collect all image containers

In [ ]:
# NOTE: the original "collect all image containers" code that lived here was accidentally
# overwritten. Restore your own version. The loaded per-dataset container lists are:
#   dic_imgs, dic_imgs_wang, merged_imgs, merged_imgs_1, dic_imgs_roan

## Before/after finetuning comparison (training data)

For each sample in `microsam_finetune_data.json`, a two-row figure comparing the base micro-sam model (before finetuning) with the Candida-finetuned model (after finetuning). Each row shows the ground-truth mask and the predicted mask overlaid on the DIC image, then the three raw decoder maps (cell probability, boundary distance, center distance). Figures are written to `thesis_figures/finetune_comparison/`.

In [ ]:
# Uses find_files_by_pattern to locate each sample's input files (as elsewhere in this
# notebook) and reads the model-tagged masks/raw maps the pipeline saved. Masks are overlaid
# on the merged image (multi-channel composites in colour, DIC-only in grayscale), which
# already carries the per-sample correct_DIC_shift. The ground-truth label is stored
# unshifted, so it is shifted the same way via an is_label ImageContainer to match the
# shifted image and the predicted masks. Raw maps are shown in their stored stack order
# [foreground, center, boundary].
import json
import numpy as np
import cv2
import tifffile
import matplotlib.pyplot as plt
from copy import deepcopy
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer

FT_FIG_DIR = Path("../../thesis_figures/finetune_comparison")
FT_FIG_DIR.mkdir(parents=True, exist_ok=True)
FT_DATA = json.loads(Path("microsam_finetune_data.json").read_text())

ROW_LABEL = {"before_finetune": "Before finetuning", "after_finetune": "After finetuning"}
MODEL_ROWS = [(k, ROW_LABEL.get(k, k), Path(v["checkpoint_path"]).stem) for k, v in MODELS.items()]
# raw stack order [foreground, center, boundary], shown in that order.
RAW_MAPS = [("Cell probability", 0), ("Center distance", 1), ("Boundary distance", 2)]
COL_TITLES = ["Ground truth", "Prediction"] + [t for t, _ in RAW_MAPS]


def _find_one(search_path, name):
    hits = find_files_by_pattern(search_path, name)
    if not hits:
        raise FileNotFoundError(f"{name} not found in {search_path}")
    return hits[0]


def _build_container(sample, cfg):
    """Locate a sample's input files and build the aligned ImageContainer
    (DIC-only, or [DIC, fluorescence, DAPI] composite as in the dataset cells)."""
    img = sample["image"]
    dic = Path(img["dic"]).expanduser()
    dic = _find_one(dic.parent, dic.name)
    if "fluorescence" in img:
        fluo = _find_one(dic.parent, Path(img["fluorescence"]).expanduser().name)
        dapi = _find_one(dic.parent, Path(img["dapi"]).expanduser().name)
        container = ImageContainer([dic, fluo, dapi], cfg)
    else:
        container = ImageContainer(dic, cfg)
    label = Path(sample["label"]).expanduser()
    label = _find_one(label.parent, label.name)
    return container, dic.stem, label


def _find_output(out_dir, name, dic_stem, prefix, kind):
    """Saved file for a model on this image: {name}_automatic_{prefix}_<date>_<hash>_{kind}.tif.
    Falls back to matching the DIC stem, since a composite's container name is not
    reproducible across sessions (its source-path order comes from a set)."""
    hits = sorted(out_dir.glob(f"{name}_automatic_{prefix}_*_{kind}.tif"))
    if not hits:
        hits = [p for p in out_dir.glob(f"*_automatic_{prefix}_*_{kind}.tif") if dic_stem in p.name]
    if not hits:
        raise FileNotFoundError(f"No {kind} for {prefix} in {out_dir} (name={name!r})")
    return max(hits, key=lambda p: p.stat().st_mtime)


def _norm(img, lo=1, hi=99):
    """Percentile-stretch to 0-1: 2D stays grayscale; (H, W, 3) is stretched per channel."""
    img = img.astype(np.float32)
    if img.ndim == 2:
        a, b = np.percentile(img, [lo, hi])
        return np.clip((img - a) / (b - a + 1e-8), 0.0, 1.0)
    out = np.zeros_like(img)
    for ch in range(img.shape[2]):
        a, b = np.percentile(img[..., ch], [lo, hi])
        out[..., ch] = np.clip((img[..., ch] - a) / (b - a + 1e-8), 0.0, 1.0)
    return out


def _match(a, hw, nearest=True):
    if a.shape[-2:] == hw:
        return a
    interp = cv2.INTER_NEAREST if nearest else cv2.INTER_LINEAR
    if a.ndim == 2:
        return cv2.resize(a, (hw[1], hw[0]), interpolation=interp)
    return np.stack([cv2.resize(a[i], (hw[1], hw[0]), interpolation=interp) for i in range(a.shape[0])])


def _overlay(ax, bg, labels, alpha=0.5, seed=0):
    """Draw an instance mask over a merged background (grayscale or RGB)."""
    ax.imshow(bg, cmap="gray" if bg.ndim == 2 else None)
    if labels is not None and labels.max() > 0:
        rng = np.random.default_rng(seed)
        colors = rng.random((int(labels.max()) + 1, 3)).astype(np.float32)
        colors[0] = 0.0
        rgba = np.zeros((*labels.shape, 4), dtype=np.float32)
        rgba[..., :3] = colors[labels]
        rgba[..., 3] = (labels > 0).astype(np.float32) * alpha
        ax.imshow(rgba)


def _shifted_label(label_path, shift, base_config):
    """Ground-truth label shifted the same way as the DIC, via an is_label ImageContainer.
    The label file is stored unshifted while the drawn masks align to the shifted DIC, so the
    label must be shifted by the same correct_DIC_shift to match the image and predictions."""
    cfg = deepcopy(base_config)
    cfg.setdefault("preprocessing", {})["correct_DIC_shift"] = shift
    return ImageContainer(label_path, cfg, is_label=True).channels[0].image_16bit


def _render_sample_panel(sample, base_config, axes, ds=2):
    """Draw one sample's before/after comparison rows into a supplied (nrow, ncol) axes grid.
    Multi-channel merges are collapsed to grayscale (channel mean) so the colored mask
    overlays stay legible, matching the generalization and tiling figures."""
    shift = sample.get("correct_DIC_shift", [0, 0])
    cfg = deepcopy(base_config)
    cfg.setdefault("preprocessing", {})["correct_DIC_shift"] = shift
    container, dic_stem, label_path = _build_container(sample, cfg)
    name = container.name
    out_dir = container.channels[0].path.parent / "microsam_outputs"

    hw = container.channels[0].image_16bit.shape[-2:]
    merged = container.merge()
    if merged.ndim == 3:
        merged = merged.mean(axis=2)
    bg = _norm(merged)[::ds, ::ds]
    gt = _match(_shifted_label(label_path, shift, base_config).astype(np.int32), hw)[::ds, ::ds]

    for r, (mkey, row_label, prefix) in enumerate(MODEL_ROWS):
        masks = _match(tifffile.imread(str(_find_output(out_dir, name, dic_stem, prefix, "masks"))).astype(np.int32), hw)[::ds, ::ds]
        raw = _match(tifffile.imread(str(_find_output(out_dir, name, dic_stem, prefix, "raw"))).astype(np.float32), hw, nearest=False)[:, ::ds, ::ds]
        _overlay(axes[r][0], bg, gt)
        _overlay(axes[r][1], bg, masks)
        for c, (title, idx) in enumerate(RAW_MAPS, start=2):
            axes[r][c].imshow(raw[idx], cmap="magma")
        for c in range(len(axes[r])):
            axes[r][c].set_xticks([]); axes[r][c].set_yticks([])
        axes[r][0].set_ylabel(row_label, fontsize=17)
    for c, title in enumerate(COL_TITLES):
        axes[0][c].set_title(title, fontsize=17)
    return dic_stem


PANEL_LETTERS = ["A", "B", "C", "D", "E", "F"]


def make_finetune_comparison_grid(samples, base_config, ds=2, dpi=200):
    """One composed 2x3 figure, one panel (A-F) per finetuning-data sample, in sample order.
    Each panel keeps the 2-row (before/after) x 5-column (GT, prediction, 3 raw maps) layout.
    Panel letters are Arial Bold black, in a dedicated white strip above each panel's content
    (never overlaid on the plots)."""
    from matplotlib.font_manager import FontProperties
    ARIAL_BOLD = FontProperties(fname="/usr/share/fonts/truetype/msttcorefonts/arialbd.ttf")

    assert len(samples) == 6, f"expected 6 samples for a 2x3 grid, got {len(samples)}"
    prow, pcol = 3, 2
    nrow, ncol = len(MODEL_ROWS), len(COL_TITLES)
    panel_w, panel_h = 2.9 * ncol, 3.3 * nrow
    label_h = 0.75

    fig = plt.figure(figsize=(panel_w * pcol, (panel_h + label_h) * prow), layout="constrained")
    outer = fig.add_gridspec(prow, pcol, hspace=0.15, wspace=0.08)

    for i, sample in enumerate(samples):
        pr, pc = divmod(i, pcol)
        cell = outer[pr, pc].subgridspec(2, 1, height_ratios=[label_h, panel_h], hspace=0.02)

        # Separate sub-axes for the letter and the image name, so their fixed-point-size
        # text never overlaps regardless of how small the label strip is in inches.
        label_gs = cell[0].subgridspec(2, 1, height_ratios=[3, 2], hspace=0.0)
        letter_ax = fig.add_subplot(label_gs[0])
        letter_ax.set_facecolor("white")
        letter_ax.axis("off")
        name_ax = fig.add_subplot(label_gs[1])
        name_ax.set_facecolor("white")
        name_ax.axis("off")

        content_gs = cell[1].subgridspec(nrow, ncol, wspace=0.05, hspace=0.04)
        axes = [[fig.add_subplot(content_gs[r, c]) for c in range(ncol)] for r in range(nrow)]
        dic_stem = _render_sample_panel(sample, base_config, axes, ds=ds)

        letter_ax.text(0.0, 0.5, PANEL_LETTERS[i], ha="left", va="center",
                      fontsize=28, fontproperties=ARIAL_BOLD, color="black")
        name_ax.text(0.5, 0.5, dic_stem, ha="center", va="center",
                      fontsize=13, color="black")

    out = FT_FIG_DIR.parent / "assembled_finetune_comparison.png"
    fig.savefig(out, dpi=dpi, bbox_inches="tight", facecolor="white")
    print(f"wrote {out}")
    plt.show()


make_finetune_comparison_grid(FT_DATA["samples"], config)

## Generalization to non-training images (before vs after finetuning)

Images loaded above but not part of the finetuning data, so they show how the finetuned model generalizes. Five are chosen across the datasets (the Roan DIC-only set is excluded because all of it is training data). Two rows, top before finetuning and bottom after, each the predicted mask overlaid on the DIC. Saved to `thesis_figures/generalization/`.

In [ ]:
# Reuses the loaded dataset lists and the helpers defined in the training-comparison cell
# (_overlay, _norm, _match, _find_output, MODEL_ROWS). Training images are dropped by DIC
# stem so only genuinely unseen images remain. The background is rebuilt from each pick's
# channel paths with an explicit correct_DIC_shift, so it does not depend on the shared
# `config` (which the dataset cells mutate) or on lazy-cache timing. It is shown in grayscale
# (composites averaged across channels) so the coloured mask overlays stay legible.
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from copy import deepcopy
from image_processing_tools.image_class.image_container import ImageContainer

GEN_FIG_DIR = Path("../../thesis_figures/generalization")
GEN_FIG_DIR.mkdir(parents=True, exist_ok=True)

_train_stems = {Path(s["image"]["dic"]).stem for s in FT_DATA["samples"]}


def _non_train(containers):
    """Loaded containers whose DIC is not one of the finetuning-data images."""
    return [c for c in containers if c.channels[0].path.stem not in _train_stems]


def _merged_bg(container, shift, base_config):
    """Full-resolution merged image with `shift` applied, collapsed to a single grayscale
    intensity so the coloured mask overlays stay legible. Multi-channel composites are
    averaged across channels (all channels still contribute); DIC-only images are already
    single-channel. Rebuilt fresh from the container's channel paths with a private config
    so it is independent of the shared config's current correct_DIC_shift and of lazy-cache
    timing. Channel order is preserved (merge iterates channels in order)."""
    cfg = deepcopy(base_config)
    cfg.setdefault("preprocessing", {})["correct_DIC_shift"] = shift
    paths = [ch.path for ch in container.channels]
    structure = paths if len(paths) > 1 else paths[0]
    merged = ImageContainer(structure, cfg).merge()
    return merged.mean(axis=2) if merged.ndim == 3 else merged


# Five non-training images spread across datasets: Eva mono, Wang hyphae, Eva
# coculture (x2), Roan coculture. correct_DIC_shift matches each dataset's cell.
GEN_PICKS = [
    (_non_train(dic_imgs)[0], [0, 0], "Hyphae Mono-culture"),
    (_non_train(dic_imgs_wang)[0], [0, 0], "Hyphae Mono-culture"),
    (_non_train(merged_imgs)[0], [5, 22], "Co-culture"),
    (_non_train(merged_imgs)[1], [5, 22], "Co-culture"),
    (_non_train(merged_imgs_1)[0], [5, 22], "Co-culture"),
]


def make_generalization_figure(picks, ds=2, dpi=200):
    nrow, ncol = len(MODEL_ROWS), len(picks)
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.0 * ncol, 3.2 * nrow),
                             squeeze=False, layout="constrained")
    for c, (container, shift, dslabel) in enumerate(picks):
        dic_path = container.channels[0].path
        out_dir = dic_path.parent / "microsam_outputs"
        name, dic_stem = container.name, dic_path.stem
        bg_full = _merged_bg(container, shift, config)
        hw = bg_full.shape[:2]
        bg = _norm(bg_full)[::ds, ::ds]
        for r, (mkey, row_label, prefix) in enumerate(MODEL_ROWS):
            mpath = _find_output(out_dir, name, dic_stem, prefix, "masks")
            masks = _match(tifffile.imread(str(mpath)).astype(np.int32), hw)[::ds, ::ds]
            _overlay(axes[r][c], bg, masks)
            axes[r][c].set_xticks([]); axes[r][c].set_yticks([])
        axes[0][c].set_title(dslabel, fontsize=17, pad=26)
        axes[0][c].text(0.5, 1.015, dic_stem, transform=axes[0][c].transAxes,
                        ha="center", va="bottom", fontsize=10)
    for r, (mkey, row_label, prefix) in enumerate(MODEL_ROWS):
        axes[r][0].set_ylabel(row_label, fontsize=17)
    fig.suptitle("Generalization to non-training images: before vs after finetuning", fontsize=18)
    fig.savefig(GEN_FIG_DIR / "generalization_before_after.png", dpi=dpi, bbox_inches="tight")
    plt.show()
    return fig


make_generalization_figure(GEN_PICKS)

## Tiling speed and quality (original model)

How tile size trades off segmentation speed against mask quality, on the base (pre-finetuning) `vit_b_lm` model so it can be shown before the finetuning section. The same five images are segmented with no tiling and with progressively smaller tiles: (1024, 128), (512, 64), (256, 32). Each segmentation is timed from scratch (its cached embedding is cleared first), and the embedding caches are kept separate per tiling setting so they never overwrite each other. Results are saved to `thesis_figures/tiling_comparison/`.

This cell runs real segmentation on the GPU, so it is the slow one in this notebook.

In [ ]:
# Runs on the ORIGINAL model. Uses the same five images as the generalization figure and the
# helpers from the cells above (_merged_bg, _norm, _match, _overlay). Masks are kept in memory
# only; save_results is not called, so nothing under the data folders is touched.
#
# The picks are rebuilt as fresh containers carrying their explicit correct_DIC_shift, so the
# live segmentation does not read the shared `config`'s current (mutated) shift value. The
# generalization plot works because it reads pre-saved masks; here the masks are computed live,
# so the container's shift must be pinned or the masks and background would misalign. The
# background is rebuilt with the same shift, so masks and image stay aligned.
import time
import shutil
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
from image_processing_tools.image_class.image_container import ImageContainer
from image_processing_tools.image_class.micro_sam_pipeline import MicroSAMPipeline

TILE_FIG_DIR = Path("../../thesis_figures/tiling_comparison")
TILE_FIG_DIR.mkdir(parents=True, exist_ok=True)

TILING_SETTINGS = [
    ("No tiling", None, None),
    ("tile 1024, halo 128", (1024, 1024), (128, 128)),
    ("tile 512, halo 64", (512, 512), (64, 64)),
    ("tile 256, halo 32", (256, 256), (32, 32)),
]


def _fresh_container(container, shift, base_config):
    """Rebuild a container from its channel paths with an explicit correct_DIC_shift, so its
    preprocessing is independent of the shared config's current (mutated) shift value."""
    cfg = deepcopy(base_config)
    cfg.setdefault("preprocessing", {})["correct_DIC_shift"] = shift
    paths = [ch.path for ch in container.channels]
    structure = paths if len(paths) > 1 else paths[0]
    return ImageContainer(structure, cfg)


# Shift-correct containers, built once so every tiling setting segments the same image.
TILE_PICKS = [(_fresh_container(c, s, config), s, dslabel) for c, s, dslabel in GEN_PICKS]

BASE_MODEL = MODELS["before_finetune"]  # original, pre-finetuning model
tiling_masks = {}   # setting label -> {dic_stem: instance mask}
tiling_times = {}   # setting label -> {dic_stem: seconds}

print(f"{'setting':22s} | {'image':45s} | {'seconds':>8s}")
for label, tile_shape, halo in TILING_SETTINGS:
    cfg = deepcopy(config)
    cfg["checkpoint_path"] = BASE_MODEL["checkpoint_path"]
    cfg["decoder_checkpoint_path"] = BASE_MODEL["decoder_checkpoint_path"]
    cfg["segmentation_mode"] = "automatic"
    cfg["tiling"] = {"tile_shape": tile_shape, "halo": halo}
    pipe = MicroSAMPipeline([c for c, _s, _l in TILE_PICKS], config=cfg)
    tiling_masks[label], tiling_times[label] = {}, {}
    for container, shift, dslabel in TILE_PICKS:
        stem = container.channels[0].path.stem
        emb = pipe._get_embedding_path(container)   # tiling-aware, unique per setting
        if emb.exists():
            shutil.rmtree(emb)                       # time a true from-scratch segmentation
        t0 = time.perf_counter()
        _key, result = pipe._run_ais_on_image(container)
        dt = time.perf_counter() - t0
        tiling_masks[label][stem] = result["masks"]
        tiling_times[label][stem] = dt
        print(f"{label:22s} | {stem:45s} | {dt:8.1f}")


def make_tiling_figure(masks_by_setting, times_by_setting, picks, ds=2, dpi=200):
    """One composed figure: (A) mask grid on the left, (B) mean timing bars on the right.
    Panel letters are Arial Bold black, in a dedicated white strip above each panel's
    content (never overlaid on the plots)."""
    from matplotlib.font_manager import FontProperties
    ARIAL_BOLD = FontProperties(fname="/usr/share/fonts/truetype/msttcorefonts/arialbd.ttf")

    labels = [s[0] for s in TILING_SETTINGS]
    nrow, ncol = len(labels), len(picks)
    fig_h = 3.2 * nrow
    label_h = 0.55  # inches of dedicated white space for the A/B letters
    fig = plt.figure(figsize=(3.0 * ncol + 6.5, fig_h), layout="constrained")
    outer = fig.add_gridspec(2, 2, height_ratios=[label_h, fig_h - label_h],
                             width_ratios=[3.0 * ncol, 6.5], hspace=0.02, wspace=0.15)

    for col, letter in ((0, "A"), (1, "B")):
        label_ax = fig.add_subplot(outer[0, col])
        label_ax.set_facecolor("white")
        label_ax.axis("off")
        label_ax.text(0.0, 0.3, letter, ha="left", va="baseline",
                      fontsize=26, fontproperties=ARIAL_BOLD, color="black")

    mask_gs = outer[1, 0].subgridspec(nrow, ncol, wspace=0.05, hspace=0.15)
    axes = np.array([[fig.add_subplot(mask_gs[r, c]) for c in range(ncol)] for r in range(nrow)])
    for c, (container, shift, dslabel) in enumerate(picks):
        stem = container.channels[0].path.stem
        bg_full = _merged_bg(container, shift, config)
        hw = bg_full.shape[:2]
        bg = _norm(bg_full)[::ds, ::ds]
        for r, label in enumerate(labels):
            masks = _match(masks_by_setting[label][stem].astype(np.int32), hw)[::ds, ::ds]
            _overlay(axes[r][c], bg, masks)
            axes[r][c].set_xticks([]); axes[r][c].set_yticks([])
        axes[0][c].set_title(dslabel, fontsize=17, pad=26)
        axes[0][c].text(0.5, 1.015, stem, transform=axes[0][c].transAxes,
                        ha="center", va="bottom", fontsize=10)
    for r, label in enumerate(labels):
        mean_t = np.mean(list(times_by_setting[label].values()))
        axes[r][0].set_ylabel(f"{label}\n(mean {mean_t:.1f}s)", fontsize=17)

    means = [np.mean(list(times_by_setting[s[0]].values())) for s in TILING_SETTINGS]
    axt = fig.add_subplot(outer[1, 1])
    axt.bar(labels, means, color="#4a7dc9")
    axt.set_ylabel("mean segmentation time (s)", fontsize=18)
    axt.tick_params(axis="both", labelsize=16)
    axt.tick_params(axis="x", rotation=25)
    for lab in axt.get_xticklabels():
        lab.set_ha("right")
    for i, m in enumerate(means):
        axt.text(i, m, f"{m:.1f}", ha="center", va="bottom", fontsize=16)

    out = TILE_FIG_DIR / "tiling_comparison.png"
    fig.savefig(out, dpi=dpi, bbox_inches="tight", facecolor="white")
    print(f"wrote {out}")
    plt.show()


make_tiling_figure(tiling_masks, tiling_times, TILE_PICKS)